In [3]:
import json
import torch
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

# 1. Load the structured chunks from Phase 1
with open("nccn_parsed_corpus.json", "r", encoding="utf-8") as f:
    saved_chunks = json.load(f)

# Use the first 100 chunks for rapid testing
corpus = [chunk["text"] for chunk in saved_chunks[:100]]
chunk_ids = [chunk["metadata"]["chunk_id"] for chunk in saved_chunks[:100]]

# 2. Initialize our winning Phase 2 Model (PubMedBERT)
print("Loading PubMedBERT...")
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model = SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO", device=device)

# Encode the corpus once to insert into Qdrant
print("Encoding corpus...")
corpus_embeddings = model.encode(corpus)

/home/creo/coding/Internship/Oncology_Research_Assistant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading PubMedBERT...


Loading weights: 100%|███████████████████████| 199/199 [00:00<00:00, 72315.59it/s]


Encoding corpus...


In [13]:
import re
import ollama
import pandas as pd

class ClinicalHybridRetriever:
    def __init__(self, corpus: list, embeddings: list, chunk_ids: list):
        self.corpus = corpus
        self.chunk_ids = chunk_ids
        
        # --- 1. Init Dense Search (Qdrant) ---
        self.qdrant = QdrantClient(location=":memory:") # In-memory DB for Jupyter
        self.qdrant.create_collection(
            collection_name="nccn_guidelines",
            vectors_config=VectorParams(size=len(embeddings[0]), distance=Distance.COSINE)
        )
        
        # Insert vectors into Qdrant
        points = [
            PointStruct(id=i, vector=embeddings[i].tolist(), payload={"text": corpus[i], "chunk_id": chunk_ids[i]})
            for i in range(len(corpus))
        ]
        self.qdrant.upsert(collection_name="nccn_guidelines", points=points)
        
        # --- 2. Init Sparse Search (BM25) ---
        tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized_corpus)


    def route_query(self, query: str) -> str:
        """Agentic routing using Llama 3 to classify clinical intent."""
        
        system_prompt = """You are a strict query classification router for an oncology clinical assistant. 
        Analyze the user's query and classify it into EXACTLY ONE of these three categories:
        
        1. 'Recency': Queries asking for the 'latest', 'recent', or 'new' trial results and evidence.
        2. 'Precision': Queries searching for a highly specific, exact identifier like a ClinicalTrial ID (e.g., NCT02296125).
        3. 'Conceptual': General clinical questions, guidelines, biomarkers (like HER2 or BARD1), and management strategies.
        
        Respond with ONLY the category word. No explanations, no punctuation."""

        try:
            response = ollama.chat(model='llama3', messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': query}
            ])
            
            # Clean the output in case the LLM gets chatty
            raw_classification = response['message']['content'].strip().strip("'").strip('"').lower()
            
            # Defensive fallback mapping
            if "recency" in raw_classification:
                return "Recency"
            elif "precision" in raw_classification:
                return "Precision"
            else:
                return "Conceptual"
                
        except Exception as e:
            print(f"LLM Routing failed, falling back to Conceptual. Error: {e}")
            return "Conceptual"


    def retrieve(self, query: str, top_k: int = 5):
        query_type = self.route_query(query)
        
        # If Recency, we would normally trigger Live APIs (Phase 5). For now, flag it.
        if query_type == "Recency":
            return {"query_type": query_type, "results": [{"text": "Trigger Phase 5: Live API (PubMed/ClinicalTrials) routing."}]}

        
        
        # --- Dense Retrieval ---
        query_vector = model.encode(query).tolist()

        dense_hits = self.qdrant.query_points(
            collection_name="nccn_guidelines",
            query=query_vector,
            limit=top_k
        ).points
        dense_results = [{"chunk_id": hit.payload["chunk_id"], "text": hit.payload["text"]} for hit in dense_hits]
        # --- Sparse Retrieval ---
        tokenized_query = query.lower().split()
        sparse_scores = self.bm25.get_scores(tokenized_query)
        
        import numpy as np
        sparse_top_indices = np.argsort(sparse_scores)[::-1][:top_k]
        sparse_results = [{"chunk_id": self.chunk_ids[i], "text": self.corpus[i]} 
                          for i in sparse_top_indices if sparse_scores[i] > 0]

        # --- RRF Merging ---
        # If precision query, we weight sparse hits heavier by letting them rank higher
        rrf_scores = {}
        for rank, item in enumerate(dense_results):
            rrf_scores[item["chunk_id"]] = {"score": 1.0 / (60 + rank), "text": item["text"]}
            
        for rank, item in enumerate(sparse_results):
            cid = item["chunk_id"]
            if cid not in rrf_scores:
                rrf_scores[cid] = {"score": 0.0, "text": item["text"]}
            # Give a slight edge to exact keywords if it's a precision query
            k_factor = 30 if query_type == "Precision" else 60
            rrf_scores[cid]["score"] += 1.0 / (k_factor + rank)

        fused = sorted(rrf_scores.values(), key=lambda x: x["score"], reverse=True)[:top_k]
        return {"query_type": query_type, "results": fused}

In [14]:
hybrid_engine = ClinicalHybridRetriever(corpus, corpus_embeddings, chunk_ids)

test_queries = [
    "What is HER2-low breast cancer?", # Conceptual
    "NCT02296125",                     # Precision
    "Latest KRAS G12C results"         # Recency
]

print("--- HYBRID ROUTING & RETRIEVAL TEST ---")
for q in test_queries:
    response = hybrid_engine.retrieve(q, top_k=1)
    print(f"\nUser Query: '{q}'")
    print(f"Routed As:  [{response['query_type']}]")
    print(f"Top Result: {response['results'][0]['text'][:150]}...")

--- HYBRID ROUTING & RETRIEVAL TEST ---

User Query: 'What is HER2-low breast cancer?'
Routed As:  [Conceptual]
Top Result: Genetic/Familial High-Risk Assessment: Breast, Ovarian, and Pancreatic...

User Query: 'NCT02296125'
Routed As:  [Precision]
Top Result: ¥ Patient advocacy...

User Query: 'Latest KRAS G12C results'
Routed As:  [Recency]
Top Result: Trigger Phase 5: Live API (PubMed/ClinicalTrials) routing....
